<a href="https://colab.research.google.com/github/Saishiva-hub/Rag-Agent-chatbot/blob/main/RAG_agent_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain-google-genai langchain-community langchain chromadb

In [ ]:
import os
import getpass
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings , ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
def setup_env():
  if not os.getenv('GOOGlE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass('Enter your API key -')

setup_env()

In [ ]:
import sys

In [ ]:
def load_and_split(filepath):
  if not os.path.exists(filepath):
    print(f'Error: File not found at {filepath}')
    sys.exit(1)

  print(f'LOADING THE DATA')
  loader = TextLoader(filepath)
  docs = loader.load()

  print(f'SPlITING THE DATA INTO CHUNKS')
  splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 200)
  splits = splitter.split_documents(docs)
  print(f'split {len(splits)} chunks')
  return splits


In [ ]:
def create_rag_chain(splits):
  print(f'Embedding -> Initialize vector store -> create rag chain')
  embeddings = GoogleGenerativeAIEmbeddings(model = 'gemini-embedding-001',task_type = 'retrieval_document')
  vectorstores = Chroma.from_documents(documents = splits , embedding=embeddings)
  Retrieval = vectorstores.as_retriever()

  llm = ChatGoogleGenerativeAI(model = 'gemini-2.5-flash')


  template = """Answer the question based only on the following context : {context}
  Question : {question}

  Helpful Answer :"""
  prompt = ChatPromptTemplate.from_template(template)

  def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

    chain = (
        {'context': retriever | format_docs , 'question':RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser
    )
    return chain

In [ ]:
setup_env()
filepath = '/content/harrypotter-Khushi.txt'

NameError: name 'setup_env' is not defined

In [ ]:
splits = load_and_split(filepath)
rag_chain = create_rag_chain(splits)

NameError: name 'load_and_split' is not defined

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
def respond(message,chat_history):
  try:
    return rag_chain.invoke(message)
  except Exception as e:
    return f'An error occured {e}'


demo = gr.ChatInterface(
    fn = respond,
    textbox = gr.Textbox(placeholder='Ask a question related to Harry Potter',container=False,scale=7)
)
demo.launch(share=True,debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7fc12b6ed7ee50911a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
